Data Prep, Milvus Creation, and VDB Retrieval (k=1000)

In [3]:
!pip install pymilvus[milvus_lite] sentence-transformers datasets tqdm -q

In [4]:
import os
import json
import torch
from tqdm import tqdm
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from pymilvus import MilvusClient
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

In [5]:
hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)
print("HuggingFace login successful")

HuggingFace login successful


In [6]:
BASE = "/kaggle/input/datasets/umagaba/retrieval"
OUT_DIR = "/kaggle/working"
DB_PATH = f"{OUT_DIR}/legal_ir.db"
COLLECTION_NAME = "legal_corpus"

def load_label_file(path):
    labels = {}
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            parts = line.split("\t")
            if len(parts) != 2: continue
            labels[parts[0]] = parts[1].split(",")
    return labels

print("Loading label files...")
train_doc_labels   = load_label_file(f"{BASE}/train_legal_docs.txt")
train_query_labels = load_label_file(f"{BASE}/train_queries.txt")
val_doc_labels     = load_label_file(f"{BASE}/val_legal_docs.txt")
val_query_labels   = load_label_file(f"{BASE}/val_queries.txt")
test_doc_labels    = load_label_file(f"{BASE}/test_legal_docs.txt")
test_query_labels  = load_label_file(f"{BASE}/test_queries.txt")
print("Loaded")

Loading label files...
Loaded


In [7]:
dataset = load_dataset("Exploration-Lab/IL-TUR", "pcr")

def extract_sentences(split):
    return {str(sample['id']): sample['text'] for sample in split}

train_doc_sentences = extract_sentences(dataset['train_candidates'])
val_doc_sentences   = extract_sentences(dataset['dev_candidates'])
test_doc_sentences  = extract_sentences(dataset['test_candidates'])
train_query_sentences = extract_sentences(dataset['train_queries'])
val_query_sentences   = extract_sentences(dataset['dev_queries'])
test_query_sentences  = extract_sentences(dataset['test_queries'])

def filter_by_roles(doc_sentences, doc_labels, keep_roles):
    filtered = {}
    for doc_id, labels in doc_labels.items():
        sents = doc_sentences.get(doc_id)
        if not sents: continue
        min_len = min(len(sents), len(labels))
        kept = [s.strip() for s, l in zip(sents[:min_len], labels[:min_len]) if l in keep_roles]
        if kept: filtered[doc_id] = " ".join(kept)
    return filtered

CORPUS_ROLES = {"Facts", "Issues", "Ratio of the decision"}
QUERY_ROLES = {"Facts", "Issues"}

train_corpus = filter_by_roles(train_doc_sentences, train_doc_labels, CORPUS_ROLES)
val_corpus   = filter_by_roles(val_doc_sentences, val_doc_labels, CORPUS_ROLES)
test_corpus  = filter_by_roles(test_doc_sentences, test_doc_labels, CORPUS_ROLES)

train_queries = filter_by_roles(train_query_sentences, train_query_labels, QUERY_ROLES)
val_queries   = filter_by_roles(val_query_sentences, val_query_labels, QUERY_ROLES)
test_queries  = filter_by_roles(test_query_sentences, test_query_labels, QUERY_ROLES)

README.md: 0.00B [00:00, ?B/s]

pcr/train_candidates-00000-of-00001.parq(…):   0%|          | 0.00/77.0M [00:00<?, ?B/s]

pcr/dev_candidates-00000-of-00001.parque(…):   0%|          | 0.00/25.1M [00:00<?, ?B/s]

pcr/test_candidates-00000-of-00001.parqu(…):   0%|          | 0.00/34.2M [00:00<?, ?B/s]

pcr/train_queries-00000-of-00001.parquet:   0%|          | 0.00/16.0M [00:00<?, ?B/s]

...

Generating dev_candidates split:   0%|          | 0/1023 [00:00<?, ? examples/s]

Generating test_candidates split:   0%|          | 0/1727 [00:00<?, ? examples/s]

Generating train_queries split:   0%|          | 0/827 [00:00<?, ? examples/s]

Generating dev_queries split:   0%|          | 0/118 [00:00<?, ? examples/s]

Generating test_queries split:   0%|          | 0/237 [00:00<?, ? examples/s]

In [8]:
full_corpus = {**train_corpus, **val_corpus, **test_corpus}
all_queries = {**train_queries, **val_queries, **test_queries}

with open(f"{OUT_DIR}/full_corpus.json", "w") as f: json.dump(full_corpus, f)
with open(f"{OUT_DIR}/all_queries.json", "w") as f: json.dump(all_queries, f)
print(f"Saved {OUT_DIR}/full_corpus.json and all_queries.json")

Saved /kaggle/working/full_corpus.json and all_queries.json


In [9]:
print("Loading embedding model...")
embed_model = SentenceTransformer("Snowflake/snowflake-arctic-embed-l")
DIM = embed_model.get_sentence_embedding_dimension()

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/107 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

In [10]:
client = MilvusClient(DB_PATH)
if client.has_collection(COLLECTION_NAME):
    client.drop_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME, dimension=DIM, metric_type="L2",
    id_field_name="id", vector_field_name="embedding", enable_dynamic_field=True,
)
index_params = client.prepare_index_params()
index_params.add_index(field_name="embedding", index_type="IVF_FLAT", metric_type="L2", params={"nlist": 2048})
client.create_index(collection_name=COLLECTION_NAME, index_params=index_params)
client.load_collection(COLLECTION_NAME)

BATCH_SIZE = 64
corpus_ids = list(full_corpus.keys())
corpus_texts = [full_corpus[d] for d in corpus_ids]

print("Embedding and inserting corpus into Milvus...")
for batch_start in tqdm(range(0, len(corpus_texts), BATCH_SIZE)):
    batch_texts = corpus_texts[batch_start : batch_start + BATCH_SIZE]
    batch_ids   = corpus_ids[batch_start : batch_start + BATCH_SIZE]
    embeddings = embed_model.encode(batch_texts, batch_size=BATCH_SIZE, show_progress_bar=False, normalize_embeddings=True).tolist()
    
    data = [
        {"id": int(bid) if bid.isdigit() else abs(hash(bid)) % (2**31), "embedding": emb, "doc_id": bid, "text": txt[:60000]}
        for bid, emb, txt in zip(batch_ids, embeddings, batch_texts)
    ]
    client.insert(collection_name=COLLECTION_NAME, data=data)

Embedding and inserting corpus into Milvus...


100%|██████████| 88/88 [11:31<00:00,  7.85s/it]


In [11]:
# retrieve k=1000 for BM25
print("Running VDB Retrieval (k=1000)...")
vdb_ranked, vdb_detailed = {}, {}

for qid, qtext in tqdm(all_queries.items(), desc="VDB Search"):
    vec = embed_model.encode([qtext], normalize_embeddings=True).tolist()
    raw = client.search(collection_name=COLLECTION_NAME, data=vec, limit=1000, output_fields=["doc_id"], search_params={"metric_type": "L2", "params": {"nprobe": 64}})
    
    results = [{"doc_id": h["entity"]["doc_id"], "distance": h["distance"], "rank": r + 1} for r, h in enumerate(raw[0])]
    vdb_ranked[qid] = [r["doc_id"] for r in results]
    vdb_detailed[qid] = results

with open(f"{OUT_DIR}/vdb_ranked_1000.json", "w") as f: json.dump(vdb_ranked, f, indent=2)
with open(f"{OUT_DIR}/vdb_detailed_1000.json", "w") as f: json.dump(vdb_detailed, f, indent=2)
print("Vector DB queries complete! Files saved to /kaggle/working/")

Running VDB Retrieval (k=1000)...


VDB Search: 100%|██████████| 978/978 [03:26<00:00,  4.74it/s]


Vector DB queries complete! Files saved to /kaggle/working/


E0411 17:16:57.252488     834 chttp2_transport.cc:1385] unix:/tmp/tmp72nlfowp_legal_ir.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms
